# PySide6 学习笔记（入门速通）

> 目标：掌握 PySide6 的安装、最小应用结构、信号与槽、布局管理、常用控件与线程协作。

PySide6 是 Qt 官方推出的 Python 绑定，让你用 Python 调用 Qt 6 的全部 API[citation:1]。与 PyQt6 相比，PySide6 由 The Qt Company 直接维护，开发过程更透明，文档以 Python 为中心，长期维护风险更低[citation:6]。它适合开发跨平台的桌面 GUI 应用，从简单工具到复杂的数据可视化界面都能胜任。

**安装**
```bash
pip install pyside6
```

> 推荐使用虚拟环境。PySide6 的 wheel 包已自带 Qt 二进制，无需单独安装 Qt[citation:7]。

In [1]:
import PySide6
from PySide6 import QtCore, QtWidgets, QtGui

print("PySide6 版本:", PySide6.__version__)
print("Qt 版本:", PySide6.QtCore.__version__)

PySide6 版本: 6.7.2
Qt 版本: 6.7.3


## 1. 最小应用与事件循环

每个 PySide6 应用都由三个要素组成：**QApplication 实例**、**至少一个控件**、**app.exec() 事件循环**。

| 步骤 | 代码 | 作用 |
|---|---|---|
| 创建应用 | `app = QApplication(sys.argv)` | 管理应用级资源，每个程序**只能有一个** |
| 创建控件 | `label = QLabel("Hello")` | 所有 UI 元素都继承自 QWidget |
| 显示控件 | `label.show()` | 控件创建后默认不可见 |
| 进入事件循环 | `app.exec()` | 阻塞运行，等待并分发用户事件 |

**`QApplication` 与 `QWidget` 的关系**：QApplication 管理应用级的事件分发和资源，QWidget 是可见的 UI 单元。没有 QApplication，QWidget 无法正常工作。

**注意**：`app.exec()` 返回退出码，通常用 `sys.exit(app.exec())` 把退出码传给操作系统[citation:2]。

In [2]:
# 最小可运行应用（取消注释运行）
import sys
from PySide6.QtWidgets import QApplication, QLabel

app = QApplication(sys.argv)
label = QLabel("Hello World!")
label.show()
sys.exit(app.exec())

print("取消注释即可看到窗口")

SystemExit: 0

d:\software_tech\anaconda3\envs\python_learn\Lib\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## 2. 信号与槽

信号与槽是 Qt 的核心通信机制：**信号**在特定事件发生时被发射，**槽**是接收信号的函数。通过 `signal.connect(slot)` 建立连接。

| 概念 | 说明 | 示例 |
|---|---|---|
| 信号 | 控件内置的事件通知 | `button.clicked`、`entry.textChanged` |
| 槽 | 接收信号的函数或方法 | 普通 Python 函数即可 |
| 连接 | 把信号和槽绑定 | `button.clicked.connect(say_hello)` |

**`@Slot()` 装饰器**：官方建议给槽函数加上 `@Slot()`，避免意外行为[citation:5]。虽然不强制，但养成习惯有助于类型检查和信号重载解析。

**信号可以携带参数**：`clicked` 信号携带 `checked` 布尔值，`textChanged` 携带新文本。槽函数的参数数量必须**不超过**信号参数数量。

**易错点**：`connect()` 接收的是**函数对象**，写 `connect(func)` 而不是 `connect(func())`，加括号会立即执行。

In [ ]:
# 信号与槽演示（取消注释运行）
# import sys
# from PySide6.QtWidgets import QApplication, QPushButton
# from PySide6.QtCore import Slot

# @Slot()
# def say_hello():
#     print("Button clicked, Hello!")

# app = QApplication(sys.argv)
# button = QPushButton("Click me")
# button.clicked.connect(say_hello)
# button.show()
# sys.exit(app.exec())

print("信号与槽代码已注释")

## 3. 布局管理

控件不指定布局就**不会自动排列**。Qt 提供多种布局管理器，自动处理控件的位置和尺寸。

| 布局类 | 排列方式 | 适用场景 |
|---|---|---|
| `QHBoxLayout` | 水平排列 | 工具栏、按钮行 |
| `QVBoxLayout` | 垂直排列 | 表单、侧边栏 |
| `QGridLayout` | 网格（行/列） | 计算器键盘、复杂表单 |
| `QFormLayout` | 两列表单 | 标签 + 输入框配对[citation:14] |

**设置布局的方式**：创建布局对象 → `addWidget()` 添加控件 → `parent.setLayout(layout)` 把布局应用到父控件。

**`QFormLayout` 的便捷方法**：`addRow("标签文本", widget)` 一行代码完成标签和输入框的配对，比手动创建 QLabel 更简洁。

**嵌套布局**：布局可以嵌套——把子布局 `addLayout()` 到父布局中，实现复杂界面。

In [ ]:
# 表单布局演示（取消注释运行）
# import sys
# from PySide6.QtWidgets import (QApplication, QWidget, QFormLayout,
#                                QLineEdit, QComboBox, QPushButton)

# app = QApplication(sys.argv)
# widget = QWidget()
# layout = QFormLayout()

# layout.addRow("用户名:", QLineEdit())
# layout.addRow("密码:", QLineEdit())
# layout.addRow("角色:", QComboBox())
# layout.addRow(QPushButton("登录"))

# widget.setLayout(layout)
# widget.show()
# sys.exit(app.exec())

print("布局代码已注释")

## 4. 常用控件

PySide6 的控件集中在 `QtWidgets` 模块。所有控件都支持 `setText`、`setEnabled`、`setStyleSheet` 等通用方法。

| 控件 | 类名 | 常用信号 |
|---|---|---|
| 标签 | `QLabel` | — |
| 按钮 | `QPushButton` | `clicked` |
| 单行输入 | `QLineEdit` | `textChanged`、`returnPressed` |
| 多行文本 | `QTextEdit` | `textChanged` |
| 下拉框 | `QComboBox` | `currentIndexChanged` |
| 复选框 | `QCheckBox` | `stateChanged` |
| 数值输入 | `QSpinBox` / `QDoubleSpinBox` | `valueChanged` |
| 表格 | `QTableWidget` | `cellClicked` |

**获取/设置值**：`QLineEdit.text()` 读文本，`setText()` 写；`QComboBox.currentText()` 读当前选项；`QSpinBox.value()` 读数。

**样式**：`setStyleSheet("background-color: #f0f0f0; font-size: 14px;")` 支持 CSS 风格的样式定制。

In [ ]:
# 控件交互演示（取消注释运行）
# import sys
# from PySide6.QtWidgets import (QApplication, QWidget, QVBoxLayout,
#                                QLineEdit, QPushButton, QLabel)

# app = QApplication(sys.argv)
# widget = QWidget()
# layout = QVBoxLayout(widget)

# entry = QLineEdit()
# label = QLabel("输入内容将显示在这里")

# def update_label(text):
#     label.setText(text or "（空）")

# entry.textChanged.connect(update_label)
# layout.addWidget(entry)
# layout.addWidget(label)
# layout.addWidget(QPushButton("确定"))

# widget.show()
# sys.exit(app.exec())

print("控件代码已注释")

## 5. 线程与后台任务

**GUI 操作必须在主线程执行**，耗时任务如果直接写在主线程中会导致界面卡死。PySide6 的标准做法是：后台任务放到 `QThread` 中，通过**信号**把结果传回主线程。

| 类/机制 | 作用 |
|---|---|
| `QThread` | 后台线程的基类，重写 `run()` 定义任务 |
| `Signal` | 自定义信号，用于跨线程通信 |
| `@Slot` | 标记接收信号的槽函数 |
| `.emit(value)` | 发射信号，触发连接的槽 |

**核心原则**：**不要在子线程中直接操作任何控件**。所有 UI 更新都通过信号回到主线程执行。

**信号的定义位置**：信号必须定义在继承 `QObject` 的类中，`Signal(str)` 表示携带一个字符串参数[citation:10]。

In [ ]:
# 线程与信号演示（取消注释运行）
# import sys, time
# from PySide6.QtCore import QThread, Signal, Slot, QObject
# from PySide6.QtWidgets import QApplication, QWidget, QVBoxLayout, QPushButton, QLabel

# class Worker(QThread):
#     result_ready = Signal(str)
#     def run(self):
#         time.sleep(2)
#         self.result_ready.emit("后台任务完成")

# class MainWindow(QWidget):
#     def __init__(self):
#         super().__init__()
#         layout = QVBoxLayout(self)
#         self.label = QLabel("点击按钮开始")
#         btn = QPushButton("耗时任务")
#         btn.clicked.connect(self.start_task)
#         layout.addWidget(self.label)
#         layout.addWidget(btn)

#     def start_task(self):
#         self.worker = Worker()
#         self.worker.result_ready.connect(self.on_result)
#         self.worker.start()

#     @Slot(str)
#     def on_result(self, msg):
#         self.label.setText(msg)

# app = QApplication(sys.argv)
# win = MainWindow()
# win.show()
# sys.exit(app.exec())

print("线程代码已注释")

## 6. 综合练习

1. 写一个简易计数器：一个标签显示数字，两个按钮“+1”和“-1”。
2. 做一个文件选择器：按钮打开 `QFileDialog`，把选中的路径显示在 QLineEdit 中。
3. 用 QThread 实现一个倒计时：从 10 开始每秒减 1，完成后弹窗提示。

**参考答案（计数器）**

In [ ]:
# 计数器（取消注释运行）
# import sys
# from PySide6.QtWidgets import (QApplication, QWidget, QVBoxLayout,
#                                QHBoxLayout, QPushButton, QLabel)

# class Counter(QWidget):
#     def __init__(self):
#         super().__init__()
#         self.value = 0
#         self.label = QLabel("0")
#         self.label.setStyleSheet("font-size: 32px;")

#         plus = QPushButton("+1")
#         minus = QPushButton("-1")
#         plus.clicked.connect(lambda: self.update(1))
#         minus.clicked.connect(lambda: self.update(-1))

#         h = QHBoxLayout()
#         h.addWidget(minus)
#         h.addWidget(plus)
#         v = QVBoxLayout(self)
#         v.addWidget(self.label)
#         v.addLayout(h)

#     def update(self, delta):
#         self.value += delta
#         self.label.setText(str(self.value))

# app = QApplication(sys.argv)
# win = Counter()
# win.show()
# sys.exit(app.exec())

print("计数器代码已注释")

## 常见坑点总结

- **忘记 `app.exec()`**：窗口创建后一闪而过，事件循环是 GUI 运行的核心。
- **`connect` 传了函数调用**：写 `connect(func)` 而不是 `connect(func())`，加括号会立即执行并返回 `None`。
- **在子线程中操作控件**：会导致崩溃或未定义行为，所有 UI 更新必须通过信号回到主线程。
- **控件没有加入布局**：不设置布局的控件不会自动排列，需要手动 `move()` 或 `setGeometry()`。
- **`QApplication` 重复创建**：一个进程只能有一个 QApplication 实例。
- **信号参数不匹配**：槽函数的参数数量不能超过信号携带的参数数量。
- **PySide6 与 PyQt6 混用**：两者是独立的包，不能在同一应用中共存，项目选型时需统一[citation:6]。